# BMC spatiotemporal cube base class

The spatiotemporal_cube is the child class of the spatial_engine base class and forms the template for any of the data cubes that will be generated by the bmc library. 
The cube class is what is referred to as an abstract base class which offers a blueprint that makes the generation of the data cubes a standardized pipeline that can be translated to different data providers. 

Operating as this blueprint, an Abstract Base Class (implemented via Python's abc module) acts as a strict contract. Because spatiotemporal_cube inherits from ABC, it cannot be instantiated directly. Instead, it guarantees that any child class—such as those built for specific data providers—must implement a required set of methods before it can function.

This architecture drives the standardized pipeline through the Template Method design pattern. The spatiotemporal_cube provides a fully realized, concrete method called ``process_cube``, which serves as the universal processing engine. However, this main engine relies entirely on placeholder "abstract" methods to navigate the specific nuances of different data providers.

Ultimately, this structure means that the heavy lifting—complex spatial physics, directory generation, logging, and core GDAL/xarray processing—only needs to be written once in the parent class. The child classes are then free to simply fill in the vendor-specific blanks.

<div align="center">
    <img src="img/process_cube_pipeline.png" alt="process_cube Pipeline Workflow" width="800px">
</div>

## 1. generate_execution_plan

### The Role of this Method
The `generate_execution_plan` method is the critical first step in the data cube pipeline. Its entire purpose is to bridge the gap between a user’s abstract configuration (like a time range and a bounding box) and the physical location of those files. 

Because every data provider organizes their assets differently, the parent `spatiotemporal_cube` outsources this search to the child class via this abstract method.

### The Strict Contract
To satisfy the parent class, your child class must return a `pandas.DataFrame`. To prevent the downstream `process_cube` engine from crashing, this DataFrame is strictly required to contain at least these three columns:

* **`level`** (str): The processing family or temporal grouping (e.g., 'TCF', 'GRA').
* **`variable`** (str): The scientific variable name (e.g., 'Tree_Cover_Density').
* **`vsi_path`** (str): The direct network URL (via `/vsicurl/`) or local system path to the exact GeoTIFF file.

### Practical example

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
import logging
from typing import Dict, Any

# Note: This method would exist inside your child class (e.g., class WEkEOCube(spatiotemporal_cube):)

def generate_execution_plan(self, recipe: Dict[str, Any], logger: logging.Logger) -> pd.DataFrame:
    """
    Translates the YAML recipe into a STAC GeoParquet query, returning a standardized asset queue.
    """
    logger.info("Loading STAC GeoParquet catalog for asset intersection...")
    
    # ==========================================
    # STEP 1: Extract Spatial & Temporal Criteria
    # ==========================================
    # Get temporal bounds
    temporal_cfg = recipe.get('temporal', {})
    start_year = temporal_cfg.get('start_year', 2018)
    end_year = temporal_cfg.get('end_year', 2019)
    
    # Get spatial bounds and build a Shapely bounding box geometry
    bbox_cfg = recipe.get('spatial', {}).get('bbox', {})
    user_bbox = box(
        bbox_cfg.get('long_min', 0), 
        bbox_cfg.get('lat_min', 0), 
        bbox_cfg.get('long_max', 0), 
        bbox_cfg.get('lat_max', 0)
    )

    # ==========================================
    # STEP 2: Load and Filter the GeoParquet Catalog
    # ==========================================
    # Get the catalog path from the YAML config
    catalog_path = recipe.get('sources', {}).get('wekeo', {}).get('catalog_path', 'stac_catalog.parquet')
    
    try:
        # Load the vectorized catalog into memory
        stac_gdf = gpd.read_parquet(catalog_path)
    except Exception as e:
        logger.error(f"Failed to load GeoParquet catalog: {e}")
        return pd.DataFrame() # Return empty to safely abort

    # Filter 1: Temporal intersection (assuming STAC has a 'datetime' column)
    stac_gdf = stac_gdf[
        (stac_gdf['datetime'].dt.year >= start_year) & 
        (stac_gdf['datetime'].dt.year <= end_year)
    ]
    
    # Filter 2: Spatial intersection
    # geopandas handles this mathematically via spatial indexing
    stac_gdf = stac_gdf[stac_gdf.intersects(user_bbox)]
    
    # ==========================================
    # STEP 3: Map Active Variables to the Catalog
    # ==========================================
    # The YAML specifies which variables the user wants (e.g., Tree_Cover_Density: true)
    tcf_datasets = recipe['sources']['wekeo']['categories']['TCF']['datasets']
    active_vars = [var for var, attrs in tcf_datasets.items() if attrs.get('include', False)]
    
    # Filter 3: Asset Variable (Assuming STAC has an 'asset_name' or 'collection' column)
    stac_gdf = stac_gdf[stac_gdf['asset_name'].isin(active_vars)]
    
    # ==========================================
    # STEP 4: Format to Satisfy the ABC Contract
    # ==========================================
    # The parent engine strictly requires 'level', 'variable', and 'vsi_path'
    execution_df = pd.DataFrame({
        'level': stac_gdf['collection'],      # e.g., 'TCF'
        'variable': stac_gdf['asset_name'],   # e.g., 'Tree_Cover_Density'
        'vsi_path': stac_gdf['href'],         # The actual cloud URL or local path
        'year': stac_gdf['datetime'].dt.year  # Metadata to parse later on the Z-axis
    })
    
    logger.info(f"STAC intersection complete. Generated plan with {len(execution_df)} valid assets.")
    
    return execution_df

## 2. resolve_target_grid

### The Role of this Method
The `resolve_target_grid` method ensures that all disparate datasets are warped onto a flawlessly aligned master canvas. While a user might arbitrarily request a specific grid and resolution in their YAML config (e.g., `target_grid: "EEA"`, `target_resolution: "500m"`), different data vendors have strict physical limitations.

This method acts as a spatial safety valve. It interprets the user's spatial configuration and maps it to a safe, mathematically supported grid framework present in the parent class's internal `GRID_REGISTRY`. It gives the child class the authority to enforce vendor-specific logic—such as stopping a user from actively corrupting atmospheric data by requesting a resolution that introduces false precision.

### The Strict Contract
To satisfy the parent class, your child class must return a single **`str`**. 

This string must be a validated dictionary key (e.g., `'EEA_1km'`, `'Global_EqualArea_100m'`) that exists within the parent `spatiotemporal_cube`'s `GRID_REGISTRY`.

### Example

In [1]:
import logging
from typing import Dict, Any

# Note: This method would exist inside your child class (e.g., class CHELSACube(spatiotemporal_cube):)

def resolve_target_grid(self, spatial_cfg: Dict[str, Any], logger: logging.Logger) -> str:
    """
    Translates user spatial configurations into a validated master grid key, 
    enforcing CHELSA's physical 1km limit.
    """
    logger.info("Resolving and validating target spatial grid...")
    
    # ==========================================
    # STEP 1: Read the User's Request
    # ==========================================
    requested_grid = spatial_cfg.get('target_grid', 'Global_WGS84') # e.g., 'EEA'
    requested_res = spatial_cfg.get('target_resolution', '1km')     # e.g., '500m'
    
    # ==========================================
    # STEP 2: Apply Vendor-Specific Safety Logic
    # ==========================================
    # We must prevent the user from downscaling 1km climate data to 500m.
    if requested_grid == "EEA" and requested_res == "500m":
        logger.warning(
            "CHELSA variables cannot be safely downscaled below 1km. "
            "Overriding requested 500m resolution to native 1km to prevent false precision."
        )
        # Force the resolution to a safe limit
        requested_res = "1km"

    # ==========================================
    # STEP 3: Construct the Parent Registry Key
    # ==========================================
    # Combine the grid and resolution to match the parent's registry format
    master_grid_key = f"{requested_grid}_{requested_res}"
    
    # Ensure this key actually exists in the parent's dictionary before passing it back
    if master_grid_key not in self.GRID_REGISTRY:
        logger.error(f"Grid key '{master_grid_key}' is not mathematically supported by the parent registry.")
        raise ValueError(f"Unsupported grid target: {master_grid_key}")
        
    logger.info(f"Target grid successfully locked to: {master_grid_key}")
    
    # ==========================================
    # STEP 4: Fulfill the ABC Contract
    # ==========================================
    return master_grid_key

In many cases, the data you are working with might be incredibly high resolution (like 10m satellite imagery), or the vendor might not have strict mathematical limits on how the data is handled. In these scenarios, the child class doesn't need to step in and play "police"—it can simply build the key directly from the user's configuration and pass it along.

Here is how a more permissive child class (for example, a `wekeo_cube` handling high-resolution land cover data) would implement the exact same method. Notice how it skips the override logic but still performs a basic safety check against the parent's registry.

In [ ]:
import logging
from typing import Dict, Any

# Note: This method would exist inside your child class (e.g., class wekeo_cube(spatiotemporal_cube):)

def resolve_target_grid(self, spatial_cfg: Dict[str, Any], logger: logging.Logger) -> str:
    """
    Translates user spatial configurations into a master grid key without enforcing 
    vendor-specific limits, trusting the user's recipe.
    """
    logger.info("Resolving target spatial grid from user configuration...")
    
    # ==========================================
    # STEP 1: Read the User's Request
    # ==========================================
    # We pull the exact values requested in the YAML recipe
    requested_grid = spatial_cfg.get('target_grid', 'Global_WGS84')
    requested_res = spatial_cfg.get('target_resolution', '100m')
    
    # ==========================================
    # STEP 2: Construct the Parent Registry Key
    # ==========================================
    # Simply combine them into the required string format
    master_grid_key = f"{requested_grid}_{requested_res}"
    
    # ==========================================
    # STEP 3: Basic Parent Safety Check
    # ==========================================
    # Even if we trust the user's resolution, we still must ensure the parent 
    # spatiotemporal_cube actually has the mathematical definition for this grid.
    if master_grid_key not in self.GRID_REGISTRY:
        logger.error(f"The requested grid '{master_grid_key}' is not defined in the master registry.")
        raise ValueError(f"Please select a supported grid and resolution combination.")
        
    logger.info(f"Target grid successfully locked to: {master_grid_key}")
    
    # ==========================================
    # STEP 4: Fulfill the ABC Contract
    # ==========================================
    return master_grid_key

Whether your child class needs to strictly enforce physical limits (like the CHELSA 1km limit) or simply pass the user's request through (like this WEkEO example), the parent class (spatiotemporal_cube) doesn't care. As long as the abstract method returns a valid string key that exists in the GRID_REGISTRY, the parent engine has exactly what it needs to execute the transform_bounds calculations and out-of-core warping.

## 3. get_resample_rule

### The Role of this Method
During the cube pipeline, the parent `process_cube` engine performs a massive out-of-core affine reprojection to warp and align raw data onto the master grid. This operation changes the spatial layout of your pixels, which forces the engine to calculate entirely new cell values. 

Because different ecological and climate variables represent fundamentally distinct physical realities, the parent engine cannot blindly guess how to interpolate these values. It outsources this decision to the child class via `get_resample_rule` to enforce strict constraints that protect physical and scientific integrity:

* **Preserving Totals & Conservation Laws:** For flux or volumetric variables like precipitation (`pr`) or runoff, standard geometric interpolations like `bilinear` or `cubic` are physically invalid because they do not guarantee mass or volume conservation across boundaries. These require statistical aggregators like `average` or `sum`.
* **Handling Categorical Data Safely:** For discrete or categorical variables (such as a `Forest_Type` land cover layer containing Class 1: Broadleaved and Class 2: Coniferous), standard pixel averaging is mathematically nonsensical. Averaging Class 1 and Class 2 would yield a value of 1.5, which does not exist or represents a completely different class.
* **The Fractional Coverage Exception:** In advanced geospatial pipelines like this library, discrete data is often converted into **fractional coverage maps** (e.g., computing the *percentage* of a pixel covered by Broadleaved forest). Once converted into a continuous percentage scale (0.0 to 1.0) representing class density, geometric averaging or arithmetic aggregation suddenly becomes highly meaningful and physically accurate.

The parent class relies on this method to determine exactly which GDAL resampling rule fits the physical boundaries of each layer before it applies the warp.

### The Strict Contract
To satisfy the parent class, your child class must evaluate the incoming variable name and return a single **`str`**.

This string must match a supported GDAL resampling keyword recognized by the processing engine (e.g., `'nearest'`, `'average'`, `'mode'`, `'bilinear'`, `'rms'`).

### Example 

#### Encoding Specific Rules: The CHELSA Implementation

To truly see the power of `get_resample_rule`, let's look at a hyper-specific implementation tailored exclusively for the CHELSA dataset. 

CHELSA provides a massive array of bioclimatic variables that span entirely different physical domains. We have continuous temperatures, volumetric precipitation totals, integer-based counts of days, and even strict categorical climate classifications. 

Here is how a dedicated `chelsa_cube` child class encodes strict, domain-specific rules to protect the in

In [ ]:
import logging


def get_resample_rule(self, variable_name: str) -> str:
    """
    Enforces strict mathematical resampling algorithms based on the precise 
    physical definitions of CHELSA bioclimatic and meteorological variables.
    """
    # ==========================================
    # 1. STRICT CATEGORICAL CLASSIFICATIONS
    # ==========================================
    # Köppen-Geiger classifications (kg0, kg1, etc.) define distinct climate zones.
    # Averaging a 'Tropical' pixel with an 'Arid' pixel corrupts the classification.
    if variable_name.startswith('kg'):
        self.logger.debug(f"Variable '{variable_name}' is a Köppen-Geiger classification. Assigning: 'mode'")
        return 'mode'
        
    # ==========================================
    # 2. VOLUMETRIC & MASS CONSERVATION
    # ==========================================
    # Precipitation (pr) and its associated Bioclimatic variables (bio12-bio19) 
    # represent absolute volumes of water. Snow Water Equivalent (swe) represents mass.
    # We use 'average' to ensure the total volume of water is conserved when altering grid cells.
    elif variable_name in ['pr', 'bio12', 'bio13', 'bio14', 'bio16', 'bio17', 'bio18', 'bio19', 'swe']:
        self.logger.debug(f"Variable '{variable_name}' represents water volume/mass. Assigning: 'average'")
        return 'average'

    # ==========================================
    # 3. DISCRETE INTEGER COUNTS (TIME)
    # ==========================================
    # Variables like Growing Season Length (gsl), Snow Cover Days (scd), and 
    # Number of Growing Days (ngd5) are discrete counts of days. 
    # Interpolating these can create mathematically impossible "fractional days" (e.g., 4.3 days of snow).
    elif variable_name in ['gsl', 'scd', 'ngd0', 'ngd5', 'ngd10', 'fgd', 'fcf']:
        self.logger.debug(f"Variable '{variable_name}' represents discrete integer days. Assigning: 'nearest'")
        return 'nearest'

    # ==========================================
    # 4. CONTINUOUS METEOROLOGICAL FIELDS
    # ==========================================
    # All remaining variables (tas, vpd, hurs, bio01-bio11) represent continuous 
    # gradients (temperature, pressure, humidity). 
    # 'bilinear' provides the safest, most scientifically accepted spatial smoothing for these fields.
    else:
        self.logger.debug(f"Variable '{variable_name}' represents a continuous physical gradient. Assigning: 'bilinear'")
        return 'bilinear'

When a user defines a highly complex resampling strategy in their YAML recipe—such as requesting four different continuous aggregations (`average`, `max`, `min`, `rms`) and a specific discrete transformation (`coverage`)—the child class needs a way to process these rules without breaking the parent class's strict signature. 

Because the parent engine's `get_resample_rule` only accepts the `variable_name` as an argument, it cannot look up the YAML recipe directly at the exact moment of warping. 

To solve this, the `wekeo_cube` utilizes the **metadata smuggling** technique we discussed earlier. During the execution plan and metadata parsing phases, the specific rule from the user's recipe is baked directly into the variable's name (e.g., `"Tree_Cover_Density__max__unknown"`). 

Here is how the `get_resample_rule` beautifully decodes those exact recipe instructions right before GDAL executes the warp:

In [ ]:
import logging

# Note: This method exists inside the wekeo_cube(spatiotemporal_cube) child class

def get_resample_rule(self, variable_name: str) -> str:
    """
    Determines the appropriate GDAL spatial resampling algorithm by decoding 
    the structurally smuggled routing name derived from the user's YAML recipe.
    """
    # ==========================================
    # STEP 1: Decode the Smuggled Recipe Instruction
    # ==========================================
    # Parse the string format using the delimiter: "Variable__Aggregation__Class"
    # Example 1: "Tree_Cover_Density__max__unknown" -> ['Tree_Cover_Density', 'max', 'unknown']
    # Example 2: "Grassland__coverage__Grassland" -> ['Grassland', 'coverage', 'Grassland']
    parts = variable_name.split("__")
    
    # Safely check if the string actually contained the smuggled metadata
    if len(parts) >= 2:
        
        # Isolate the aggregation rule (the second element) and standardize to lowercase
        agg_rule = parts[1].lower()
        
        # ==========================================
        # STEP 2: Handle the 'Discrete: Coverage' Rule
        # ==========================================
        # The user requested 'coverage' for discrete data in their YAML.
        # To downsample categorical masks (like Land Cover) into continuous fractional 
        # percentages, we must use the arithmetic mean to conserve physical spatial area.
        if agg_rule == 'coverage':
            return "average"
            
        # ==========================================
        # STEP 3: Handle the 'Continuous' List Rules
        # ==========================================
        # The user requested a list of continuous stats in their YAML: [average, max, min, rms].
        # These requested statistical aggregations natively match their GDAL C++ string names.
        if agg_rule in ['max', 'min', 'average', 'rms']:
            return agg_rule
            
    # ==========================================
    # STEP 4: Safety Fallback
    # ==========================================
    # If the variable lacks a defined rule or the string was malformed, default to "nearest".
    # This prevents categorical classes from being accidentally corrupted by floating-point math.
    return "nearest"

### 4. parse_metadata

### The Role of this Method
When the core engine fetches raw Cloud-Optimized GeoTIFFs from remote storage, the resulting arrays are fundamentally "flat" 2D grids (representing only X and Y coordinates). They have no awareness of time, categorical groupings, or climate scenarios.

The `parse_metadata` method is responsible for breathing multi-dimensional life into these flat arrays. It extracts contextual metadata from the execution plan (like the year, or a specific climate model) and structurally injects it into the array, expanding it from 2D into 3D or 4D. 


Crucially, this method also serves as a defense mechanism against the underlying GDAL C++ processing engine. Because GDAL natively strips away complex Python `xarray` coordinates during spatial reprojection, child classes often use this step to "smuggle" critical metadata by temporarily baking it directly into the variable's string name.

### The Strict Contract
To satisfy the parent class, your child class must return a two-element **`tuple`**:
1.  **`level`** (str): The processing family grouping string (e.g., 'GRA', 'climatologies') used by the parent to route the array into the correct output cube.
2.  **`da`** (xarray.DataArray): The structurally augmented DataArray, safely renamed and expanded along the correct Z-axis.

In [ ]:
import pandas as pd
import xarray as xr
from typing import Tuple

# Note: This method would exist inside your child class (e.g., class wekeo_cube(spatiotemporal_cube):)

def parse_metadata(self, row: pd.Series, da: xr.DataArray) -> Tuple[str, xr.DataArray]:
    """
    Injects temporal axes and structurally encodes categorical metadata into 
    the variable's string name to survive GDAL processing.
    """
    # 1. Extract the parent processing family
    level = row['level']
    
    # 2. Lock the observation year to a standard January 1st datetime for the Z-axis
    year = int(row['year'])
    time_coord = pd.to_datetime(f"{year}-01-01")
    
    # 3. Sanitize redundant dimensions 
    # (Squeeze the default 'band' dimension to prevent out-of-core writing crashes)
    if 'band' in da.dims:
        da = da.squeeze('band').drop_vars('band', errors='ignore')
        
    # Expand the 2D array into a 3D temporal slice (time, y, x)
    structured_da = da.expand_dims(time=[time_coord])
    
    # ==========================================
    # METADATA SMUGGLING (THE GDAL WORKAROUND)
    # ==========================================
    aggregation = row.get('aggregation', 'unknown')
    layer_class = row.get('layer_class', 'unknown')
    
    # We bake the metadata into the core variable name so GDAL doesn't delete it.
    # Example Output: "Forest_Type__coverage__Broadleaved_forest"
    structured_da.name = f"{row['variable']}__{aggregation}__{layer_class}"
    
    return level, structured_da

### 5. apply_multi_index

#### The Role of this Method

This is the final operational step in the parent process_cube pipeline. After GDAL finishes mathematically warping all the spatial data, it returns a flattened xarray.Dataset.

The apply_multi_index method acts as the "unpacker" to the parse_metadata method's "packer". It allows the child class to read the smuggled metadata strings, formally promote them back into true multi-dimensional coordinates, and seamlessly concatenate independent arrays back into a formalized, scientifically pristine N-dimensional data cube. For highly complex datasets (like CHELSA CMIP6 climate ensembles), this is also where independent metadata arrays are bundled into a formal Pandas MultiIndex.

#### The Strict Contract

To satisfy the parent class, your child class must return a finalized xarray.Dataset. It receives the level string and the fully warped (but dimensionally flat) dataset as inputs.

In [ ]:
def apply_multi_index(self, level: str, dataset: xr.Dataset) -> xr.Dataset:
    """
    Unpacks the smuggled string metadata post-warp and expands the flattened 
    variables back into a formalized N-dimensional data cube.
    """
    grouped_arrays = {}
    
    # ==========================================
    # STEP 1: Decode & Expand Dimensions
    # ==========================================
    for temp_var_name, da in dataset.data_vars.items():
        
        # Crack open the smuggled string: "Variable__Aggregation__Class"
        parts = str(temp_var_name).split("__")
        base_var = parts[0]
        agg_val = parts[1] if len(parts) > 1 else 'unknown'
        class_val = parts[2] if len(parts) > 2 else 'unknown'
        
        # Expand along a new Z-axis depending on the physical data type
        if agg_val == 'coverage':
            # Discrete fractional masks gain a 'layer_class' dimension
            da = da.expand_dims(layer_class=[class_val])
        else:
            # Continuous variables gain an 'aggregation' dimension
            da = da.expand_dims(aggregation=[agg_val])
        
        # Restore the clean scientific variable name
        da.name = base_var
        
        # Group the decoded slices
        if base_var not in grouped_arrays:
            grouped_arrays[base_var] = []
        grouped_arrays[base_var].append(da)
        
    # ==========================================
    # STEP 2: Concatenate the Final Cube
    # ==========================================
    final_dataset = xr.Dataset()
    
    for base_var, arrays in grouped_arrays.items():
        # Dynamically determine the correct axis to glue the slices together
        concat_dim = 'layer_class' if 'layer_class' in arrays[0].dims else 'aggregation'
        
        # Smash the individual 4D slices together into a unified variable
        final_dataset[base_var] = xr.concat(arrays, dim=concat_dim)
        
    return final_dataset